# 02 — Banking Fraud Detection: Data Preparation

Continúa desde [`01_eda.ipynb`](./01_eda.ipynb). Este notebook traduce los hallazgos del EDA en un dataset listo para modelar:

1. **Deduplica antes de dividir train/test** — el EDA mostró que dividir primero arriesgaba filtrar duplicados exactos entre ambos conjuntos.
2. Construye dos features simples (`hour_of_day`, `log_amount`) a partir de `Time` y `Amount`.
3. Hace un split train/test **cronológico** (el dataset ya viene ordenado por `Time`), igual que en project 01.
4. **No** toca `V1`–`V28` con reglas de outliers — el EDA mostró que ahí vive la señal de fraude.
5. Deja una demostración diagnóstica de las técnicas de desbalance (class weight, SMOTE) que se compararán formalmente en `03_modeling.ipynb`, sin persistir un dataset sobremuestreado gigante en disco.

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/creditcard.csv'
TRAIN_OUT = '../data/processed_train.csv'
TEST_OUT = '../data/processed_test.csv'

In [2]:
df = pd.read_csv(RAW_PATH)
print('Filas crudas:', df.shape)

Filas crudas: (284807, 31)


## 1. Deduplicar antes de dividir

Se elimina cualquier fila exactamente duplicada (todas las columnas iguales, incluida `Class`), conservando la primera aparición. Esto se hace **antes** del split para que ningún par duplicado quede repartido entre train y test.

In [3]:
n_before = len(df)
fraud_before = df['Class'].sum()

df = df.drop_duplicates(keep='first').reset_index(drop=True)

n_after = len(df)
fraud_after = df['Class'].sum()

print(f'Filas: {n_before} -> {n_after} (-{n_before - n_after})')
print(f'Fraudes: {fraud_before} -> {fraud_after} (-{fraud_before - fraud_after})')
print(f'Tasa de fraude: {fraud_before / n_before:.4%} -> {fraud_after / n_after:.4%}')

Filas: 284807 -> 283726 (-1081)
Fraudes: 492 -> 473 (-19)
Tasa de fraude: 0.1727% -> 0.1667%


## 2. Feature engineering

Dos features simples, directamente motivadas por el EDA:

- **`hour_of_day`**: hora del día (0–23) derivada de `Time` — el EDA mostró una tasa de fraude visiblemente más alta de madrugada.
- **`log_amount`**: `log(1 + Amount)` — `Amount` está muy sesgado a la derecha (ver EDA sección 7); el log ayuda al modelo lineal (baseline) sin perjudicar a los modelos de árbol. Se mantiene `Amount` original también, por si el modelo de árbol prefiere la escala cruda.

In [4]:
df['hour_of_day'] = (df['Time'] % 86400) // 3600
df['log_amount'] = np.log1p(df['Amount'])

df[['Time', 'hour_of_day', 'Amount', 'log_amount']].head()

,Time,hour_of_day,Amount,log_amount
0,0.0,0.0,149.62,5.014760
1,0.0,0.0,2.69,1.305626
2,1.0,0.0,378.66,5.939276
3,1.0,0.0,123.50,4.824306
4,2.0,0.0,69.99,4.262539


**Nota sobre `V1`–`V28`:** no se aplica ninguna transformación ni recorte de outliers — quedan tal cual las entrega el dataset, consistente con la decisión del EDA (sección 8).

**Nota sobre escalado:** `Time` y `Amount`/`log_amount` no están en la misma escala que `V1`–`V28` (que ya vienen centradas por el PCA del proveedor). El escalado se deja para `03_modeling.ipynb`, dentro de un `Pipeline` de scikit-learn ajustado únicamente sobre train — mismo criterio que en project 01, para no filtrar estadísticas del test set.

## 3. Split train/test cronológico

El dataset ya está ordenado por `Time` (verificado en el EDA). Se separa 80% más antiguo → train, 20% más reciente → test, igual criterio que en project 01: en producción, un modelo de fraude siempre predice sobre transacciones que todavía no ocurrieron.

In [5]:
cutoff_time = df['Time'].quantile(0.8)

train_df = df[df['Time'] < cutoff_time].copy()
test_df = df[df['Time'] >= cutoff_time].copy()

print(f'Corte en Time = {cutoff_time:.0f}s (~{cutoff_time / 3600:.1f}h)')
print(f'Train: {len(train_df)} filas | fraude: {train_df["Class"].sum()} ({train_df["Class"].mean():.4%})')
print(f'Test:  {len(test_df)} filas | fraude: {test_df["Class"].sum()} ({test_df["Class"].mean():.4%})')

Corte en Time = 145234s (~40.3h)


Train: 226980 filas | fraude: 399 (0.1758%)
Test:  56746 filas | fraude: 74 (0.1304%)


Ambos conjuntos conservan una tasa de fraude similar (~0.13–0.18%) y un volumen razonable de casos positivos (399 en train, 74 en test) — suficiente para entrenar y evaluar, aunque siguen siendo pocos casos en términos absolutos.

## 4. Diagnóstico de técnicas de desbalance (sin persistir)

El EDA marcó tres técnicas a evaluar en el modelado: **class weights**, **SMOTE** y **Isolation Forest** (no supervisado). Acá solo se deja un diagnóstico rápido de las dos primeras — la comparación formal de desempeño ocurre en `03_modeling.ipynb`. No se guarda un CSV sobremuestreado: SMOTE se recalcula al vuelo en el notebook de modelado, ajustado únicamente sobre el split de train de cada corrida, para no perder trazabilidad de qué filas son sintéticas.

In [6]:
X_train_preview = train_df.drop(columns=['Class'])
y_train_preview = train_df['Class']

neg, pos = (y_train_preview == 0).sum(), (y_train_preview == 1).sum()
scale_pos_weight = neg / pos
print(f'class weight (negativas/positivas) sugerido para XGBoost: scale_pos_weight = {scale_pos_weight:.1f}')

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train_preview, y_train_preview)
print(f'SMOTE — antes: {y_train_preview.value_counts().to_dict()} | después: {y_res.value_counts().to_dict()}')

class weight (negativas/positivas) sugerido para XGBoost: scale_pos_weight = 567.9


SMOTE — antes: {0: 226581, 1: 399} | después: {0: 226581, 1: 226581}


Con un desbalance de ~568:1, `scale_pos_weight` sería un ajuste enorme — en `03_modeling.ipynb` se comparará contra SMOTE (que genera ejemplos sintéticos de fraude por interpolación) y contra Isolation Forest (que no necesita ninguna de las dos, al ser no supervisado). Ninguna técnica se descarta de antemano; se comparan con PR-AUC en el notebook de modelado.

## 5. Guardado de datasets procesados

Se guardan **deduplicados, con las dos features nuevas, sin escalar y sin resamplear** — el resampleo y el escalado quedan dentro del pipeline de modelado, no horneados en el CSV. No se versionan en git (excluidos por `.gitignore`, igual que el CSV crudo).

In [7]:
train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

print('Guardado:', TRAIN_OUT, train_df.shape)
print('Guardado:', TEST_OUT, test_df.shape)

Guardado: ../data/processed_train.csv (226980, 33)
Guardado: ../data/processed_test.csv (56746, 33)


## 6. Conclusiones de la fase de Data Preparation

1. **Deduplicado antes del split:** 1.081 filas exactas eliminadas (19 de fraude), evitando fuga de información por duplicación entre train y test.
2. **Dos features nuevas:** `hour_of_day` (patrón horario detectado en el EDA) y `log_amount` (corrige el sesgo de `Amount`).
3. **`V1`–`V28` intactas:** sin recorte de outliers, por decisión explícita del EDA.
4. **Split cronológico** 80/20 por `Time`: train con 399 fraudes, test con 74 — evalúa predicción hacia adelante, no interpolación.
5. **Desbalance:** se deja diagnosticado (`scale_pos_weight` ≈ 568, SMOTE balancea a 226.581/226.581) pero **no resuelto acá** — la comparación de class weights vs. SMOTE vs. Isolation Forest es responsabilidad de `03_modeling.ipynb`, con PR-AUC como criterio (no accuracy).
6. **Escalado diferido:** `Time`/`Amount` se escalan recién dentro del pipeline de modelado, ajustado solo sobre train.

**Siguiente paso:** `03_modeling.ipynb` — comparar Logistic Regression, XGBoost e Isolation Forest bajo las distintas estrategias de desbalance, con PR-AUC y curva precision-recall como criterio principal.